# Lab 1 - Supervised Fine-Tuning: contract review that cites its evidence

## Notebook 2 - Fine-tune with a SageMaker Training job

We now adapt **NVIDIA Nemotron 3 Nano 4B** to the checklist using **LoRA**, run as a
SageMaker AI **Training job**: a container, a script, and three user input channels (train, val and config).

### What runs where

| | |
|---|---|
| model | Hugging Face id, pulled from the Hub inside the job |
| training code | `scripts/train.py` (TRL `SFTTrainer` + PEFT LoRA) |
| configuration | `args.yaml`, written below and mounted as a channel |
| container | SageMaker PyTorch DLC, `image_uris.retrieve` |
| data | the S3 prefixes from notebook 1, as channels |
| output | `model.tar.gz` in S3 |

The whole configuration is a file you can read. Notebook 3 deploys the job's artifact to an
endpoint, and notebook 4 evaluates that endpoint.


In [ ]:
%load_ext autoreload
%autoreload 2

from importlib.metadata import version
from pathlib import Path

# Expected versions come from the constraints file that requirements.txt installs.
pins = dict(line.strip().split("==") for line in Path("../constraints-notebook.txt").read_text().splitlines()
            if line.startswith(("sagemaker-train==", "sagemaker-core==")))
for package, expected in pins.items():
    actual = version(package)
    if actual != expected:
        raise RuntimeError(f"{package}=={actual}; install requirements.txt and restart the kernel (expected {expected}).")

#### Setup and dependencies

The same boilerplate as notebook 1: execution role, default bucket, bucket prefix and the
boto3 clients. `role` is what the Training job runs as, and `bucket_name` /
`default_prefix` rebuild the data paths notebook 1 wrote to.

`MLFLOW_APP_ARN` in `config.py` is an optional override. If empty, setup first reuses the uniquely matching tagged stack app, ahead of any named duplicate. Only if none matches does it reuse `MLFLOW_APP_NAME` in this account/Region or create it with the execution role and notebook bucket's `mlflow-artifacts` prefix. Ambiguity stops setup. Override creation settings with `MLFLOW_ROLE_ARN` and `MLFLOW_ARTIFACT_STORE_URI`. Setup waits for readiness before training, without requiring a domain/tracking server or changing defaults. Invalid configured ARNs and setup failures stop training. The resolved ARN goes into the inline `args.yaml` below.


In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
from config import resolve_mlflow_app
mlflow_app_arn = resolve_mlflow_app(
    sm_client, role_arn=role,
    artifact_store_uri=f"s3://{sess.default_bucket()}/mlflow-artifacts",
)
print(f"MLflow App: {mlflow_app_arn}")
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")


### Where the data is

Notebook 1 uploaded `train` and `val` and printed their URIs. Nothing was registered, so
this notebook rebuilds the same two paths from `config.py` rather than looking anything up
- which is also the check that both notebooks agree on where the data lives. If the
`head_object` calls below fail, notebook 1 has not been run against this bucket.

`TRAIN_JOB_PREFIX` comes from `config.py` too. It is the job's `base_job_name`, and
notebook 3 uses the same constant to find the finished job, so the two cannot drift.


In [ ]:
import os

from config import BASE_MODEL_ID, DATA_PREFIX, TRAIN_JOB_PREFIX

input_path = f"{default_prefix}/{DATA_PREFIX}" if default_prefix else DATA_PREFIX

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"
train_config_s3_path = f"s3://{bucket_name}/{input_path}/config/args.yaml"

# Fail here, cheaply, rather than 3 minutes into a job that cannot find its data.
for uri in (train_dataset_s3_path, val_dataset_s3_path):
    key = uri.split(f"{bucket_name}/", 1)[1]
    size = s3_client.head_object(Bucket=bucket_name, Key=key)["ContentLength"]
    print(f"{uri}  ({size:,} bytes)")

print(f"\nbase model:      {BASE_MODEL_ID}")
print(f"job base name:   {TRAIN_JOB_PREFIX}")

# Exported for the %%bash cell below.
os.environ["model_id"] = BASE_MODEL_ID
os.environ["mlflow_uri"] = mlflow_app_arn
os.environ["mlflow_experiment_name"] = "nemotron-3-nano-4b-contractnli-sft"


### Track the run in MLflow

`train.py` logs loss, learning rate, eval loss, system metrics and parameters to the standalone app reused or created during setup. It never changes a domain or account default. The execution role needs tracking permissions on that app.

Open **MLflow** in Studio or use the temporary app link below, then select experiment `nemotron-3-nano-4b-contractnli-sft`. Metrics appear after container startup, dependency installation and model loading, not immediately after submission. `mlflow_log_model: false` avoids uploading a second checkpoint; SageMaker still writes the merged artifact to S3.

In [ ]:
print(f"MLflow App: {mlflow_app_arn}")
print(f"Experiment: {os.environ['mlflow_experiment_name']}")

### The recipe

`train.py` parses its configuration with TRL's `TrlParser`, which reads one YAML file
holding both the script's own arguments and every `SFTConfig` field. The cell below writes
that file, and it is the whole training configuration - LoRA rank, learning rate, sequence
length, precision, which modules get adapters. It is then uploaded to its own `config`
channel, so SageMaker mounts it at `/opt/ml/input/data/config/args.yaml` - the single
hyperparameter the job is given.

`${model_id}`, `${mlflow_uri}` and `${mlflow_experiment_name}` are substituted by the shell
from the environment variables exported above, so the base model is set once in `config.py`
and flows through to the job.

Four settings are specific to this architecture and would be wrong if copied from a dense
model:

- **`target_modules`** - Nemotron-H is a Mamba2-Transformer hybrid: 21 Mamba2 layers, 4
  attention, 17 MLP, so only 4 of 42 layers have `q/k/v/o_proj` at all. `in_proj` **is**
  trainable - the mixer calls it as a module - but `out_proj` is **not**: the fused Mamba
  kernel reads `out_proj.weight` as a raw tensor, so an adapter there never receives a
  gradient. That is 21 dead adapters and a DDP *"parameters that didn't receive grad"*
  error; NVIDIA's own NeMo recipe for this family excludes it for the same reason. And
  `gate_proj` does not exist, because this MLP is ReLU² rather than gated, so a
  Llama-style list silently matches nothing for that entry.
- **`load_in_4bit: false`** - 4-bit is reported not to work on this architecture, and at
  7.95 GB of bf16 weights it buys nothing.
- **`pad_token: <unk>`** - the tokenizer defines no pad token and the usual fallback is
  eos. That is wrong under completion-only loss: the collator labels every pad position
  `-100`, so padding with `<|im_end|>` would mask the real turn terminator and the model
  would never learn to stop. `<unk>` is never emitted by this chat template.
- **`eos_token: <|im_end|>`** - `config.json` declares `eos_token_id: 2` (`</s>`), which
  this chat template never produces. Without the override the merged checkpoint ships with
  a stop token it cannot emit.


In [ ]:
%%bash

cat > ./args.yaml <<EOF
# ---------------------------------------------------------------- ScriptArguments
model_id: "${model_id}"
mlflow_uri: "${mlflow_uri}"
mlflow_experiment_name: "${mlflow_experiment_name}"
# Metrics and parameters only. true also uploads the merged model to MLflow and
# registers it, a second copy of the ~8 GB checkpoint that is already in S3.
mlflow_log_model: false

# Channel mount points: SageMaker copies each InputData channel to
# /opt/ml/input/data/<channel>/ before train.py runs.
train_dataset_path: "/opt/ml/input/data/train/dataset.jsonl"
val_dataset_path: "/opt/ml/input/data/val/dataset.jsonl"

# prompt + completion columns. TRL turns on completion-only loss by itself for this
# format, so the contract in the prompt is context and only the JSON is supervised.
dataset_format: "prompt_completion"

# trust_remote_code is deliberately unset: transformers implements nemotron_h natively, so
# train.py resolves it to the native class. NVIDIA's remote modeling code declares no
# _supports_flash_attn / _supports_sdpa flags (eager only, transformers #43352).
attn_implementation: "flash_attention_2"
torch_dtype: "bfloat16"
cast_parameters_to_uniform_dtype: false

# bf16 LoRA, not QLoRA.
load_in_4bit: false
use_peft: true
merge_weights: true

lora_r: 32
lora_alpha: 64
lora_dropout: 0.05

# Do NOT replace with "all-linear": it pulls in 21 dead Mamba out_proj adapters (the fused
# kernel takes out_proj.weight as a raw tensor, so they get no gradient) and matches
# nothing for gate_proj, which this ReLU^2 MLP does not have.
target_modules:
  - "q_proj"        # 4 attention layers
  - "k_proj"
  - "v_proj"
  - "o_proj"
  - "up_proj"       # 17 MLP layers
  - "down_proj"
  - "in_proj"       # 21 Mamba2 layers

# The tokenizer defines no pad token, and config.json's eos_token_id (2, </s>) is never
# emitted by the chat template.
pad_token: "<unk>"
eos_token: "<|im_end|>"

# Stop once eval_loss stops improving. Only takes effect with more than one epoch.
early_stopping: true

# ---------------------------------------------------------------- SFTConfig
# /opt/ml/model is what SageMaker uploads to S3. With merge_weights above, the artifact is
# a full merged checkpoint vLLM can serve directly - no adapter to apply at load time.
output_dir: "/opt/ml/model"

# Measured with this model's own tokenizer in notebook 1. Records over it are truncated,
# not dropped, and the tail is short.
max_length: 8192

# packing must stay off: for prompt/completion records TRL derives the completion mask from
# the token-length difference, and packing destroys that boundary.
packing: false
completion_only_loss: true

num_train_epochs: 1
per_device_train_batch_size: 1
# Defaults to 8 regardless of the train batch size, which OOMs at max_length 8192.
per_device_eval_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
# transformers 5.x merged warmup_ratio into warmup_steps: an int is a step count,
# a float in [0, 1) is a ratio of total training steps. `warmup_ratio` no longer
# exists, and TrlParser aborts on unknown config keys.
warmup_steps: 0.1
lr_scheduler_type: "cosine"
max_grad_norm: 1.0
weight_decay: 0.0

bf16: true
gradient_checkpointing: true

# Safe only because out_proj is excluded above - every adapter receives a gradient.
ddp_find_unused_parameters: false

logging_strategy: "steps"
logging_steps: 5
eval_strategy: "epoch"
# Must match eval_strategy when load_best_model_at_end is on.
save_strategy: "epoch"
save_total_limit: 1
seed: 42
EOF

echo "wrote args.yaml"
cat ./args.yaml


In [ ]:
s3_client.upload_file("args.yaml", bucket_name, f"{input_path}/config/args.yaml")
os.remove("./args.yaml")

print(f"training config uploaded to:\n  {train_config_s3_path}")


### The training container

The SageMaker PyTorch Deep Learning Container, resolved for this region and instance type.
It ships PyTorch and CUDA; `scripts/requirements.txt` installs everything else - including
`transformers`, `trl` and `peft` at pinned versions - when the job starts.

The job lists five single-GPU instance types as **instance preferences**, in order:

| type | GPU | GPU memory | RAM |
|---|---|---|---|
| `ml.g5.xlarge` | A10G | 24 GB | 16 GB |
| `ml.g5.2xlarge` | A10G | 24 GB | 32 GB |
| `ml.g6.2xlarge` | L4 | 24 GB | 32 GB |
| `ml.g6e.2xlarge` | L40S | 48 GB | 64 GB |
| `ml.g6.xlarge` | L4 | 24 GB | 16 GB |

SageMaker launches the job on the first type with capacity. The account needs training job quota
for every type in the list, otherwise the job is rejected. On every type the effective batch is
`per_device_train_batch_size x gradient_accumulation_steps` = 1 x 4 = 4.


In [ ]:
from sagemaker.core import image_uris

instance_types = [
    "ml.g5.xlarge",
    "ml.g5.2xlarge",
    "ml.g6.2xlarge",
    "ml.g6e.2xlarge",
    "ml.g6.xlarge",
]
instance_count = 1
print(f"instance preferences: {instance_types}")

# The same GPU image serves every type in the list.
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.8.0",
    instance_type=instance_types[0],
    image_scope="training",
)

print(image_uri)


### Configure the trainer

`ModelTrainer` launches the Training job. Its pieces:

- **`SourceCode`** - the local `./scripts` directory is uploaded and unpacked in the
  container. `requirements.txt` is installed first, then `entry_script` runs.
- **`Compute`** - the instance preferences and count, plus `keep_alive_period_in_seconds`. A non-zero
  value keeps the instance in a **warm pool** after the job ends, so a relaunch within that
  window skips both the capacity wait and the container pull. Set it to `0` to release the
  instance immediately.
- **`Torchrun()`** - launches one process per GPU. On this one-GPU instance that is a single
  process; on a multi-GPU instance it becomes a DDP run with no other change.
- **`hyperparameters`** - a single entry, the in-container path to the recipe. Everything
  else lives in the YAML.
- **`OutputDataConfig`** - where the merged model is uploaded. Because `merge_weights: true`
  and `output_dir: /opt/ml/model`, the artifact is a **full merged checkpoint**, not an
  adapter, so vLLM can serve it directly in notebook 3.
- **`CheckpointConfig`** - mid-training checkpoints, synced to S3 so a spot interruption or
  a restart does not lose the run.

Note where the output lands: `s3://<bucket>/[<prefix>/]<TRAIN_JOB_PREFIX>/`, and SageMaker
appends `<full-job-name>/output/model.tar.gz` under it. Notebook 3 needs that artifact, and
it reads the exact URI from `DescribeTrainingJob` rather than rebuilding this string.


In [ ]:
from sagemaker.core.shapes import InstancePreference
from sagemaker.core.training.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.distributed import Torchrun
from sagemaker.train.model_trainer import ModelTrainer

source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    entry_script="train.py",
)

# Seconds the instance stays warm after the job. Billed while warm.
KEEP_ALIVE_SECONDS = 0

compute_configs = Compute(
    instance_preferences=[InstancePreference(instance_type=t) for t in instance_types],
    instance_count=instance_count,
    keep_alive_period_in_seconds=KEEP_ALIVE_SECONDS,
)

output_path = (f"s3://{bucket_name}/{default_prefix}/{TRAIN_JOB_PREFIX}"
               if default_prefix else f"s3://{bucket_name}/{TRAIN_JOB_PREFIX}")


# Reduces CUDA memory fragmentation from the Mamba mixers.
runtime_environment = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}

model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    environment=runtime_environment,
    base_job_name=TRAIN_JOB_PREFIX,
    compute=compute_configs,
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    hyperparameters={
        # The only hyperparameter: the recipe's path inside the container. TrlParser
        # reads the rest from the YAML mounted on the `config` channel below.
        "config": "/opt/ml/input/data/config/args.yaml"
    },
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoint", local_path="/opt/ml/checkpoints"
    ),
    role=role,
    sagemaker_session=sess,
)

print(f"training output: {output_path}")

### The input channels

Each `InputData` becomes a directory in the container: `channel_name="train"` arrives at
`/opt/ml/input/data/train/`. Those paths are what `args.yaml` points `train_dataset_path`
and `val_dataset_path` at, and the `config` channel is where the recipe itself comes from.


In [ ]:
from sagemaker.core.training.configs import InputData

data = [
    InputData(channel_name="train", data_source=train_dataset_s3_path),
    InputData(channel_name="val", data_source=val_dataset_s3_path),
    InputData(channel_name="config", data_source=train_config_s3_path),
]

for channel in data:
    print(f"{channel.channel_name:7s} -> /opt/ml/input/data/{channel.channel_name}/")
    print(f"{'':7s}    {channel.data_source}")


### Launch

`wait=False` returns immediately.

**How long.** At 423 records and an effective batch of 4, one epoch is 106 optimizer
steps at about 8 s each on `ml.g5.2xlarge`, so roughly 18 minutes. A reference run on
`ml.g5.2xlarge`; the other types take a different time:

| job status | time |
|---|---|
| Pending (capacity wait, not billable) | ~5 min |
| Downloading (container pull) | ~4 min |
| Training (pip install, model download, 1 epoch, evaluation, merge) | ~21 min |
| Uploading | ~2 min |

About 34 minutes end to end, 27 of them billable.

The recipe trains for one epoch. `early_stopping: true` only acts when `num_train_epochs` is
higher: it stops once `eval_loss` fails to improve by 0.01 for 3 consecutive evaluations, then
reloads the best checkpoint.


In [ ]:
model_trainer.train(input_data_config=data, wait=False, logs=False)
# sagemaker-train stores the created resource; train() returns None.
training_job = model_trainer._latest_training_job
if training_job is None:
    raise RuntimeError("No TrainingJob was created. Check the SDK version and session.")
TRAINING_JOB_NAME = training_job.training_job_name
print(f"Launched: {TRAINING_JOB_NAME}")

### Follow the run

The job is running. Two places to watch it:

- **MLflow**: select experiment `nemotron-3-nano-4b-contractnli-sft` for loss, learning rate, eval loss and system metrics. Startup latency varies; refresh after training begins.
- **Training logs**: follow the CloudWatch link below or choose **Monitor > View logs** on the Training job details page. Log streams appear after the container starts.

The MLflow link is pre-signed and valid for 5 minutes. Re-run the cell for a fresh one.

In [ ]:
# Reuse this exact name after a kernel restart; do not select another user's latest job.
print(f"training job: {TRAINING_JOB_NAME}\n")

console = f"https://{region}.console.aws.amazon.com"
print("job details:")
print(f"{console}/sagemaker/home?region={region}#/jobs/{TRAINING_JOB_NAME}\n")
print("training logs:")
print(f"{console}/cloudwatch/home?region={region}#logsV2:log-groups/log-group/"
      f"$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3D{TRAINING_JOB_NAME}\n")

if os.environ["mlflow_uri"]:
    url = sm_client.create_presigned_mlflow_app_url(
        Arn=os.environ["mlflow_uri"], ExpiresInSeconds=300)["AuthorizedUrl"]
    print(f"MLflow, experiment {os.environ['mlflow_experiment_name']}:\n{url}")
else:
    print("MLflow tracking is off for this run.")

### Wait for the job

Polls every 60 seconds until the job ends. `Completed` means the merged model is in S3 and
notebook 3 can deploy it. Stopping this cell does not stop the job.

In [ ]:
import time

started = time.monotonic()
while True:
    job = sm_client.describe_training_job(TrainingJobName=TRAINING_JOB_NAME)
    status, secondary = job["TrainingJobStatus"], job["SecondaryStatus"]
    print(f"[{(time.monotonic() - started) / 60:5.1f} min] {status} / {secondary}")
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(60)

selected = job["ResourceConfig"].get("SelectedInstanceType", "not selected yet")
print(f"\ninstance type: {selected}")

if status == "Completed":
    print(f"artifact: {job['ModelArtifacts']['S3ModelArtifacts']}")
else:
    reason = job.get('FailureReason') or 'No failure reason reported; inspect CloudWatch logs.'
    raise RuntimeError(f"{TRAINING_JOB_NAME}: {status} / {secondary}: {reason}")

### What you built

A merged bf16 checkpoint in S3 - base weights with the LoRA adapter already applied, plus
the tokenizer - packaged as `model.tar.gz` under the job's output path.

Notebook 3 finds it by asking `DescribeTrainingJob` for the last completed job whose name
starts with `TRAIN_JOB_PREFIX`.

Continue to **notebook 3** to deploy it to a SageMaker real-time endpoint. Notebook 4
evaluates the deployed endpoint.
